<a href="https://colab.research.google.com/github/korkutanapa/DCASE2025TASK2/blob/main/ORJ_DCASE_EVALUATOR_AND_OFFICIAL_CODE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import shutil

CONTENT_DIR = "/content"

for name in os.listdir(CONTENT_DIR):
    path = os.path.join(CONTENT_DIR, name)

    try:
        if os.path.islink(path) or os.path.isfile(path):
            os.remove(path)
        elif os.path.isdir(path):
            shutil.rmtree(path)
    except Exception as e:
        print(f"Could not delete {path}: {e}")

print("✅ /content cleaned.")

✅ /content cleaned.


In [ ]:
# @title
# -*- coding: utf-8 -*-
"""
DCASE 2025 Task 2
FIXED MACHINE-SPECIFIC FEATURES + Euclidean kNN(k=10) + OFFICIAL EVALUATOR
============================================================================

"""
from __future__ import annotations

import os
import re
import glob
import shutil
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =============================================================================
# 0. DEPENDENCIES
# =============================================================================

for package, import_name in [
    ("openpyxl", "openpyxl"),
    ("scikit-learn", "sklearn"),
]:
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

BASE_DIR = os.environ.get("TDA_BASE_DIR", "/content")

MACHINE_TYPES = [
    "AutoTrash",
    "BandSealer",
    "CoffeeGrinder",
    "HomeCamera",
    "Polisher",
    "ScrewFeeder",
    "ToyPet",
    "ToyRCCar",
]

TEAM_NAME = "METU"

KNN_K = 10
EPS = 1e-12
N_JOBS = -1
DECISION_QUANTILE = 0.99

SYSTEM_NAME = "FIXED_MACHINE_SPECIFIC_KNN10"

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "dcase68_fixed_oracle_features_knn10_outputs",
)
SUBMISSION_ROOT = os.path.join(
    BASE_DIR,
    "dcase68_fixed_oracle_features_knn10_submission",
)
TEAMS_ROOT = os.path.join(SUBMISSION_ROOT, "teams")

EVALUATOR_DIR = os.path.join(
    BASE_DIR,
    "dcase2025_task2_evaluator",
)
EVALUATOR_REPO = (
    "https://github.com/nttcslab/dcase2025_task2_evaluator.git"
)

# =============================================================================
# 2. FIXED MACHINE-SPECIFIC FEATURE SUBSETS
# =============================================================================

FEATURES_BY_MACHINE = {
    "AutoTrash": [
        "H1_betti_max",
        "H0_landscape_layer2_max",
        "H0_birth_skew",
        "H0_death_min",
        "H0_birth_max",
        "H0_landscape_layer5_max",
        "H1_mean_birth_death_ratio",
    ],

    "BandSealer": [
        "H0_top3_share",
        "H0_top1_share",
        "H0_death_min",
        "H1_pimage_min",
        "H0_birth_kurtosis",
    ],

    "CoffeeGrinder": [
        "H0_landscape_auc",
        "H0_lifetime_skew",
        "H0_pimage_entropy",
        "H1_q25_lifetime",
        "H1_to_H0_num_points_ratio",
        "H1_weighted_midlife_std",
        "H0_betti_num_peaks",
        "H0_landscape_layer3_max",
        "H1_std_birth_death_ratio",
        "H1_normalized_persistence_entropy",
        "H0_death_min",
        "H1_pimage_min",
        "H0_birth_max",
        "H0_death_q25",
        "H1_birth_max",
        "H1_tail_share_q90",
        "H1_tail_share_q95",
        "H1_mean_birth_death_ratio",
    ],

    "HomeCamera": [
        "H0_landscape_auc",
        "H0_lifetime_skew",
        "H1_q25_lifetime",
        "H0_landscape_layer3_max",
        "H0_median_lifetime",
        "H0_pimage_energy",
        "H1_silhouette_entropy",
        "H0_q75_lifetime",
    ],

    "Polisher": [
        "H1_min_lifetime",
        "H0_mean_birth_death_ratio",
        "H1_std_birth_death_ratio",
        "H0_q75_lifetime",
    ],

    "ScrewFeeder": [
        "H0_landscape_layer5_auc",
        "H1_death_var",
        "H0_betti_max",
        "H1_minus_H0_entropy",
        "H0_top1_share",
        "H0_median_lifetime",
    ],

    "ToyPet": [
        "H1_birth_iqr",
        "H1_max_lifetime",
        "H0_birth_skew",
        "H0_death_skew",
        "H0_death_min",
        "H1_landscape_layer2_max",
        "H0_betti_l1",
        "H0_num_points",
        "H0_landscape_layer4_max",
        "H1_mean_birth_death_ratio",
    ],

    "ToyRCCar": [
        "H0_pimage_entropy",
        "H1_death_min",
        "H1_min_midlife",
        "H0_betti_max",
        "H0_betti_num_peaks",
        "H1_max_lifetime",
        "H1_minus_H0_entropy",
        "H0_landscape_entropy",
        "H1_silhouette_entropy",
        "H0_q75_lifetime",
        "H1_betti_std",
        "H1_persistence_entropy",
    ],
}

# Safety checks
assert set(FEATURES_BY_MACHINE.keys()) == set(MACHINE_TYPES)

# =============================================================================
# 3. FILE HELPERS
# =============================================================================

def latest_file(files):
    files = [f for f in sorted(set(files)) if os.path.isfile(f)]
    if not files:
        return None
    return max(files, key=os.path.getmtime)


def find_train_file(machine: str):
    files = []
    for pattern in [
        os.path.join(
            BASE_DIR,
            f"cubical_mel_tda_features_{machine}_thr*.xlsx",
        ),
        os.path.join(
            BASE_DIR,
            "**",
            f"cubical_mel_tda_features_{machine}_thr*.xlsx",
        ),
    ]:
        files.extend(glob.glob(pattern, recursive=True))
    return latest_file(files)


def find_test_file(machine: str):
    files = []
    for pattern in [
        os.path.join(
            BASE_DIR,
            f"cubical_mel_tda_features_{machine}*.xlsx",
        ),
        os.path.join(
            BASE_DIR,
            "**",
            f"cubical_mel_tda_features_{machine}*.xlsx",
        ),
    ]:
        files.extend(glob.glob(pattern, recursive=True))

    files = [
        p for p in files
        if "_thr" not in os.path.basename(p).lower()
        and "_train" not in os.path.basename(p).lower()
    ]

    return latest_file(files)


def normalize_file_id(v) -> str:
    s = str(v).replace("\\", "/").strip()
    s = os.path.basename(s)

    if s.lower() == "nan":
        return s

    if (
        not s.lower().endswith(".wav")
        and re.match(r"^section_\d\d_\d+$", s)
    ):
        s = s + ".wav"

    return s


def get_file_ids(df: pd.DataFrame) -> np.ndarray:
    if "file_id" in df.columns:
        return (
            df["file_id"]
            .astype(str)
            .map(normalize_file_id)
            .to_numpy()
        )

    for col in [
        "file_path",
        "filename",
        "path",
        "wav_path",
    ]:
        if col in df.columns:
            return (
                df[col]
                .astype(str)
                .map(normalize_file_id)
                .to_numpy()
            )

    raise ValueError(
        "No file_id/file_path/filename/path/wav_path column found."
    )


# =============================================================================
# 4. DOMAIN / PREPROCESSING HELPERS
# =============================================================================

def normalize_domain_value(v) -> str:
    s = str(v).strip().lower()

    if "target" in s or s in {"t", "tgt"}:
        return "target"

    if "source" in s or s in {"s", "src"}:
        return "source"

    return "unknown"


def infer_train_domains(df: pd.DataFrame) -> np.ndarray:
    domains = np.full(
        len(df),
        "unknown",
        dtype=object,
    )

    lower_to_real = {
        str(c).lower(): c
        for c in df.columns
    }

    for candidate in [
        "domain",
        "domain_label",
        "source_target",
        "source/target",
        "data_domain",
    ]:
        if candidate in lower_to_real:
            col = lower_to_real[candidate]

            parsed = (
                df[col]
                .astype(str)
                .map(normalize_domain_value)
                .to_numpy(dtype=object)
            )

            known = parsed != "unknown"
            domains[known] = parsed[known]

    text = pd.Series(
        "",
        index=df.index,
        dtype=str,
    )

    for col in [
        "file_id",
        "file_path",
        "filename",
        "path",
        "wav_path",
    ]:
        if col in df.columns:
            text = (
                text
                + " "
                + df[col].astype(str).str.lower()
            )

    unknown = domains == "unknown"
    src = (
        text.str.contains(
            "source",
            regex=False,
            na=False,
        )
        .to_numpy()
    )
    domains[unknown & src] = "source"

    unknown = domains == "unknown"
    tgt = (
        text.str.contains(
            "target",
            regex=False,
            na=False,
        )
        .to_numpy()
    )
    domains[unknown & tgt] = "target"

    return domains


def numeric_column(
    df: pd.DataFrame,
    feature: str,
) -> np.ndarray:

    x = (
        pd.to_numeric(
            df[feature],
            errors="coerce",
        )
        .to_numpy(dtype=float)
    )

    x[~np.isfinite(x)] = np.nan
    return x


def robust_center_scale(
    x_source: np.ndarray,
):
    finite = x_source[
        np.isfinite(x_source)
    ]

    if len(finite) == 0:
        raise ValueError(
            "No finite source-normal values."
        )

    center = float(
        np.median(finite)
    )

    q25, q75 = np.quantile(
        finite,
        [0.25, 0.75],
    )

    iqr = float(q75 - q25)

    mad = float(
        1.4826
        * np.median(
            np.abs(finite - center)
        )
    )

    std = float(
        np.std(finite)
    )

    for scale in [
        iqr,
        mad,
        std,
    ]:
        if (
            np.isfinite(scale)
            and scale > EPS
        ):
            return center, float(scale)

    return center, 1.0


# =============================================================================
# 5. MACHINE DATA PREPARATION
# =============================================================================

def prepare_machine(machine: str):
    train_file = find_train_file(machine)
    test_file = find_test_file(machine)

    if train_file is None:
        raise FileNotFoundError(
            f"{machine}: train *_thr*.xlsx not found."
        )

    if test_file is None:
        raise FileNotFoundError(
            f"{machine}: test xlsx not found."
        )

    train_df = pd.read_excel(train_file)
    test_df = pd.read_excel(test_file)

    selected_features = FEATURES_BY_MACHINE[machine]

    missing_train = [
        f for f in selected_features
        if f not in train_df.columns
    ]

    missing_test = [
        f for f in selected_features
        if f not in test_df.columns
    ]

    if missing_train or missing_test:
        raise ValueError(
            f"{machine}: selected features missing. "
            f"train_missing={missing_train}, "
            f"test_missing={missing_test}"
        )

    domains = infer_train_domains(
        train_df
    )

    if np.any(domains == "unknown"):
        raise ValueError(
            f"{machine}: unknown TRAIN domains = "
            f"{int(np.sum(domains == 'unknown'))}"
        )

    src_mask = (
        domains == "source"
    )

    tgt_mask = (
        domains == "target"
    )

    n_source = int(
        np.sum(src_mask)
    )

    n_target = int(
        np.sum(tgt_mask)
    )

    train_cols = []
    test_cols = []
    scaling_rows = []

    for feature in selected_features:
        tr = numeric_column(
            train_df,
            feature,
        )

        te = numeric_column(
            test_df,
            feature,
        )

        # Fit preprocessing ONLY on source-normal.
        center, scale = robust_center_scale(
            tr[src_mask]
        )

        tr = np.where(
            np.isfinite(tr),
            tr,
            center,
        )

        te = np.where(
            np.isfinite(te),
            te,
            center,
        )

        tr_z = (
            tr - center
        ) / scale

        te_z = (
            te - center
        ) / scale

        train_cols.append(tr_z)
        test_cols.append(te_z)

        scaling_rows.append({
            "machine": machine,
            "feature": feature,
            "source_center": center,
            "source_scale": scale,
        })

    X_train = (
        np.column_stack(train_cols)
        .astype(float)
    )

    X_test = (
        np.column_stack(test_cols)
        .astype(float)
    )

    # Pooled known-normal bank:
    # 990 source-normal + 10 target-normal
    X_bank = np.vstack([
        X_train[src_mask],
        X_train[tgt_mask],
    ])

    file_ids = get_file_ids(
        test_df
    )

    return {
        "machine": machine,
        "train_file": train_file,
        "test_file": test_file,
        "selected_features": selected_features,
        "n_features": len(selected_features),
        "n_source": n_source,
        "n_target": n_target,
        "X_bank": X_bank,
        "X_test": X_test,
        "file_ids": file_ids,
        "scaling_rows": scaling_rows,
    }


# =============================================================================
# 6. kNN(k=10) ANOMALY SCORING
# =============================================================================

def fit_knn_and_score(
    X_bank: np.ndarray,
    X_test: np.ndarray,
):
    if len(X_bank) <= KNN_K:
        raise ValueError(
            "Normal bank smaller than k."
        )

    # +1 for leave-one-out train score
    nn = NearestNeighbors(
        n_neighbors=KNN_K + 1,
        metric="euclidean",
        n_jobs=N_JOBS,
    )

    nn.fit(X_bank)

    # Leave-one-out normal score for optional decision threshold.
    train_dist, _ = nn.kneighbors(
        X_bank,
        n_neighbors=KNN_K + 1,
    )

    train_scores = (
        train_dist[:, 1:KNN_K + 1]
        .mean(axis=1)
    )

    # Test anomaly score.
    test_dist, _ = nn.kneighbors(
        X_test,
        n_neighbors=KNN_K,
    )

    test_scores = (
        test_dist
        .mean(axis=1)
    )

    threshold = float(
        np.quantile(
            train_scores,
            DECISION_QUANTILE,
        )
    )

    decisions = (
        test_scores > threshold
    ).astype(int)

    return (
        test_scores,
        decisions,
        threshold,
        train_scores,
    )


# =============================================================================
# 7. SAVE DCASE SUBMISSION FILES
# =============================================================================

def save_prediction_files(
    machine,
    file_ids,
    anomaly_score,
    decision_result,
):
    system_dir = os.path.join(
        TEAMS_ROOT,
        TEAM_NAME,
        SYSTEM_NAME,
    )

    os.makedirs(
        system_dir,
        exist_ok=True,
    )

    score_path = os.path.join(
        system_dir,
        f"anomaly_score_{machine}_section_00_test.csv",
    )

    decision_path = os.path.join(
        system_dir,
        f"decision_result_{machine}_section_00_test.csv",
    )

    pd.DataFrame({
        0: file_ids,
        1: anomaly_score,
    }).to_csv(
        score_path,
        index=False,
        header=False,
    )

    pd.DataFrame({
        0: file_ids,
        1: decision_result,
    }).to_csv(
        decision_path,
        index=False,
        header=False,
    )

    return score_path, decision_path



# =============================================================================
# 8. MACHINE-LEVEL OFFICIAL-LOGIC METRICS
# =============================================================================

def safe_hmean(values):
    values = np.asarray(values, dtype=float)
    values = np.maximum(values, np.finfo(float).eps)
    return float(len(values) / np.sum(1.0 / values))


def read_two_column_csv(path, value_name):
    df = pd.read_csv(
        path,
        header=None,
        names=["file_id", value_name],
        dtype={0: str},
    )
    df["file_id"] = df["file_id"].map(normalize_file_id)
    return df


def load_official_ground_truth(machine: str, test_ids: np.ndarray):
    """
    Align the test rows with the official DCASE evaluation ground truth.

    Official evaluator convention:
        anomaly label: 0=normal, 1=anomaly
        domain label : 0=source, 1=target
    """
    ensure_official_evaluator()

    gt_path = os.path.join(
        EVALUATOR_DIR,
        "ground_truth_data",
        f"ground_truth_{machine}_section_00_test.csv",
    )

    domain_path = os.path.join(
        EVALUATOR_DIR,
        "ground_truth_domain",
        f"ground_truth_{machine}_section_00_test.csv",
    )

    attr_path = os.path.join(
        EVALUATOR_DIR,
        "ground_truth_attributes",
        f"ground_truth_{machine}_section_00_test.csv",
    )

    if not os.path.isfile(gt_path):
        raise FileNotFoundError(gt_path)

    if not os.path.isfile(domain_path):
        raise FileNotFoundError(domain_path)

    gt = read_two_column_csv(
        gt_path,
        "label",
    )

    dm = read_two_column_csv(
        domain_path,
        "domain",
    )

    gt_map = dict(
        zip(
            gt["file_id"],
            gt["label"].astype(int),
        )
    )

    dm_map = dict(
        zip(
            dm["file_id"],
            dm["domain"].astype(int),
        )
    )

    # Optional alias mapping used by the official evaluator.
    alias_to_gt = {}

    if os.path.isfile(attr_path):
        try:
            attr = pd.read_csv(
                attr_path,
                header=None,
                dtype=str,
            )

            if attr.shape[1] >= 2:
                for _, row in attr.iterrows():
                    gt_id = normalize_file_id(
                        row.iloc[0]
                    )

                    alias = str(
                        row.iloc[1]
                    ).strip()

                    if not alias.lower().endswith(".wav"):
                        alias += ".wav"

                    alias = normalize_file_id(
                        alias
                    )

                    alias_to_gt[alias] = gt_id

        except Exception:
            pass

    resolved = []
    missing = []

    for fid in test_ids:
        fid = normalize_file_id(
            fid
        )

        candidate = fid

        if (
            candidate not in gt_map
            and candidate in alias_to_gt
        ):
            candidate = alias_to_gt[
                candidate
            ]

        if (
            candidate not in gt_map
            or candidate not in dm_map
        ):
            missing.append(
                fid
            )

        resolved.append(
            candidate
        )

    if missing:
        raise ValueError(
            f"{machine}: {len(missing)} test file IDs could not be matched "
            f"to official ground truth. Examples: {missing[:10]}"
        )

    y_true = np.asarray(
        [
            gt_map[x]
            for x in resolved
        ],
        dtype=int,
    )

    y_domain = np.asarray(
        [
            dm_map[x]
            for x in resolved
        ],
        dtype=int,
    )

    return (
        y_true,
        y_domain,
    )


def calculate_machine_official_metrics(
    machine: str,
    file_ids: np.ndarray,
    anomaly_scores: np.ndarray,
):
    """
    Mirrors the official evaluator's per-machine AUC/pAUC logic.

    Source AUC:
        source-domain NORMAL samples + ALL anomalous samples

    Target AUC:
        target-domain NORMAL samples + ALL anomalous samples

    pAUC:
        all test samples, max_fpr=0.1
    """
    y_true, y_domain = load_official_ground_truth(
        machine,
        file_ids,
    )

    scores = np.asarray(
        anomaly_scores,
        dtype=float,
    )

    source_mask = (
        (y_domain == 0)
        | (y_true != 0)
    )

    target_mask = (
        (y_domain == 1)
        | (y_true != 0)
    )

    auc_source = float(
        roc_auc_score(
            y_true[source_mask],
            scores[source_mask],
        )
    )

    auc_target = float(
        roc_auc_score(
            y_true[target_mask],
            scores[target_mask],
        )
    )

    pauc = float(
        roc_auc_score(
            y_true,
            scores,
            max_fpr=0.1,
        )
    )

    auc_all = float(
        roc_auc_score(
            y_true,
            scores,
        )
    )

    machine_hm = safe_hmean(
        [
            auc_source,
            auc_target,
            pauc,
        ]
    )

    return {
        "machine": machine,
        "n_features": len(
            FEATURES_BY_MACHINE[machine]
        ),
        "AUC_all": auc_all,
        "AUC_source": auc_source,
        "AUC_target": auc_target,
        "pAUC_0.1": pauc,
        "HM_source_target_pAUC": machine_hm,
    }


# =============================================================================
# 9. OFFICIAL EVALUATOR
# =============================================================================

def ensure_official_evaluator():
    evaluator_script = os.path.join(
        EVALUATOR_DIR,
        "dcase2025_task2_evaluator.py",
    )

    if os.path.isfile(evaluator_script):
        return

    if os.path.isdir(EVALUATOR_DIR):
        shutil.rmtree(EVALUATOR_DIR)

    print(
        "\nCloning OFFICIAL DCASE 2025 Task 2 evaluator..."
    )

    subprocess.check_call([
        "git",
        "clone",
        EVALUATOR_REPO,
        EVALUATOR_DIR,
    ])


def run_official_evaluator():
    ensure_official_evaluator()

    evaluator_teams = os.path.join(
        EVALUATOR_DIR,
        "teams",
    )

    result_dir = os.path.join(
        EVALUATOR_DIR,
        "teams_result",
    )

    additional_dir = os.path.join(
        EVALUATOR_DIR,
        "teams_additional_result",
    )

    for path in [
        evaluator_teams,
        result_dir,
        additional_dir,
    ]:
        if os.path.isdir(path):
            shutil.rmtree(path)

    shutil.copytree(
        TEAMS_ROOT,
        evaluator_teams,
    )

    cmd = [
        sys.executable,
        "dcase2025_task2_evaluator.py",
        "--teams_root_dir",
        "./teams",
        "--result_dir",
        "./teams_result",
        "--additional_result_dir",
        "./teams_additional_result",
        "--dir_depth",
        "2",
        "--out_all",
        "True",
    ]

    print(
        "\n"
        + "=" * 130
    )
    print(
        "RUNNING OFFICIAL DCASE EVALUATOR"
    )
    print(
        "=" * 130
    )

    subprocess.run(
        cmd,
        cwd=EVALUATOR_DIR,
        check=True,
    )

    official_path = os.path.join(
        additional_dir,
        "teams_official_score.csv",
    )

    paper_path = os.path.join(
        additional_dir,
        "teams_official_score_paper.csv",
    )

    if not os.path.isfile(
        official_path
    ):
        raise FileNotFoundError(
            official_path
        )

    df = pd.read_csv(
        official_path
    )

    df = df.rename(
        columns={
            df.columns[0]: "System"
        }
    )

    if "official score" in df.columns:
        df = (
            df.sort_values(
                "official score",
                ascending=False,
            )
            .reset_index(drop=True)
        )

    compact_cols = [
        c for c in [
            "System",
            "official score",
            "arithmetic mean",
            "harmonic mean (source)",
            "harmonic mean (target)",
        ]
        if c in df.columns
    ]

    compact = df[
        compact_cols
    ].copy()

    compact.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "official_score_compact.csv",
        ),
        index=False,
    )

    df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "official_score_full.csv",
        ),
        index=False,
    )

    if os.path.isfile(
        paper_path
    ):
        pd.read_csv(
            paper_path
        ).to_csv(
            os.path.join(
                OUTPUT_DIR,
                "official_score_paper.csv",
            ),
            index=False,
        )

    print(
        "\n"
        + "=" * 130
    )
    print(
        "OFFICIAL SCORE RANKING -- COMPACT"
    )
    print(
        "=" * 130
    )
    print(
        compact.to_string(
            index=False
        )
    )

    return compact


# =============================================================================
# 10. MAIN
# =============================================================================

def clean_outputs():
    for path in [
        OUTPUT_DIR,
        SUBMISSION_ROOT,
    ]:
        if os.path.isdir(path):
            shutil.rmtree(path)

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True,
    )

    os.makedirs(
        TEAMS_ROOT,
        exist_ok=True,
    )


def main():
    print(
        "=" * 130
    )
    print(
        "DCASE 2025 TASK 2 -- FIXED MACHINE-SPECIFIC FEATURES + kNN(k=10)"
    )
    print(
        "=" * 130
    )

    print()

    print()
    print(
        f"kNN k                 : {KNN_K}"
    )
    print(
        "Distance              : Euclidean"
    )
    print(
        "Anomaly score         : mean distance to 10 nearest normal samples"
    )
    print(
        "Normal bank           : 990 source-normal + 10 target-normal"
    )
    print(
        "Scaling               : SOURCE-normal robust median/IQR"
    )
    print(
        "Score direction       : larger = more anomalous"
    )

    clean_outputs()
    ensure_official_evaluator()

    feature_rows = []
    scaling_rows = []
    score_rows = []
    machine_metrics_rows = []

    for idx, machine in enumerate(
        MACHINE_TYPES,
        start=1,
    ):
        print(
            "\n"
            + "=" * 130
        )
        print(
            f"[{idx}/{len(MACHINE_TYPES)}] {machine}"
        )
        print(
            "=" * 130
        )

        data = prepare_machine(
            machine
        )

        print(
            f"train source={data['n_source']} "
            f"target={data['n_target']} "
            f"test={len(data['X_test'])}"
        )

        print(
            f"selected features = {data['n_features']}"
        )

        for rank, feature in enumerate(
            data["selected_features"],
            start=1,
        ):
            print(
                f"  {rank:2d}. {feature}"
            )

            feature_rows.append({
                "machine": machine,
                "rank": rank,
                "feature": feature,
            })

        scaling_rows.extend(
            data["scaling_rows"]
        )

        (
            test_scores,
            decisions,
            threshold,
            train_scores,
        ) = fit_knn_and_score(
            data["X_bank"],
            data["X_test"],
        )

        save_prediction_files(
            machine=machine,
            file_ids=data["file_ids"],
            anomaly_score=test_scores,
            decision_result=decisions,
        )

        machine_metrics = calculate_machine_official_metrics(
            machine=machine,
            file_ids=data["file_ids"],
            anomaly_scores=test_scores,
        )

        machine_metrics_rows.append(
            machine_metrics
        )

        print()
        print("MACHINE OFFICIAL-LOGIC METRICS")
        print(
            f"  AUC all       = {machine_metrics['AUC_all']:.4f}"
        )
        print(
            f"  AUC source    = {machine_metrics['AUC_source']:.4f}"
        )
        print(
            f"  AUC target    = {machine_metrics['AUC_target']:.4f}"
        )
        print(
            f"  pAUC@0.1      = {machine_metrics['pAUC_0.1']:.4f}"
        )
        print(
            f"  HM(S,T,pAUC)  = {machine_metrics['HM_source_target_pAUC']:.4f}"
        )

        score_rows.append({
            "machine": machine,
            "n_features": data["n_features"],
            "decision_q99_threshold": threshold,
            "normal_score_median": float(
                np.median(train_scores)
            ),
            "normal_score_q95": float(
                np.quantile(
                    train_scores,
                    0.95,
                )
            ),
            "normal_score_q99": float(
                np.quantile(
                    train_scores,
                    0.99,
                )
            ),
            "test_score_min": float(
                np.min(test_scores)
            ),
            "test_score_median": float(
                np.median(test_scores)
            ),
            "test_score_max": float(
                np.max(test_scores)
            ),
        })

    pd.DataFrame(
        feature_rows
    ).to_csv(
        os.path.join(
            OUTPUT_DIR,
            "fixed_features_by_machine.csv",
        ),
        index=False,
    )

    pd.DataFrame(
        scaling_rows
    ).to_csv(
        os.path.join(
            OUTPUT_DIR,
            "source_scaling_parameters.csv",
        ),
        index=False,
    )

    pd.DataFrame(
        score_rows
    ).to_csv(
        os.path.join(
            OUTPUT_DIR,
            "knn10_score_diagnostics.csv",
        ),
        index=False,
    )

    machine_metrics_df = pd.DataFrame(
        machine_metrics_rows
    )

    machine_metrics_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "machine_official_metrics.csv",
        ),
        index=False,
    )

    print(
        "\n"
        + "=" * 130
    )
    print(
        "MACHINE-BY-MACHINE RESULTS"
    )
    print(
        "=" * 130
    )

    print(
        machine_metrics_df.to_string(
            index=False,
            formatters={
                "AUC_all": lambda x: f"{x:.4f}",
                "AUC_source": lambda x: f"{x:.4f}",
                "AUC_target": lambda x: f"{x:.4f}",
                "pAUC_0.1": lambda x: f"{x:.4f}",
                "HM_source_target_pAUC": lambda x: f"{x:.4f}",
            },
        )
    )

    # Useful aggregate diagnostics.
    hm_source = safe_hmean(
        machine_metrics_df["AUC_source"].to_numpy(float)
    )
    hm_target = safe_hmean(
        machine_metrics_df["AUC_target"].to_numpy(float)
    )
    hm_pauc = safe_hmean(
        machine_metrics_df["pAUC_0.1"].to_numpy(float)
    )

    exact_overall_hm = safe_hmean(
        np.concatenate(
            [
                machine_metrics_df["AUC_source"].to_numpy(float),
                machine_metrics_df["AUC_target"].to_numpy(float),
                machine_metrics_df["pAUC_0.1"].to_numpy(float),
            ]
        )
    )

    print()
    print(
        f"HM source AUC   = {hm_source:.6f}"
    )
    print(
        f"HM target AUC   = {hm_target:.6f}"
    )
    print(
        f"HM pAUC@0.1     = {hm_pauc:.6f}"
    )
    print(
        f"Combined HM     = {exact_overall_hm:.6f}"
    )

    compact = run_official_evaluator()

    print(
        "\n"
        + "=" * 130
    )
    print(
        "DONE"
    )
    print(
        "=" * 130
    )



    print()
    print(
        "Outputs:"
    )
    print(
        " ",
        OUTPUT_DIR,
    )
    print(
        "Machine results CSV:",
        os.path.join(OUTPUT_DIR, "machine_official_metrics.csv"),
    )

    return compact


if __name__ == "__main__":
    main()